In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim
from pyspark.sql import functions as F

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
df = spark.read.parquet("/user/hive/warehouse/sclsepb4532_raw.db/customers", header=True, inferSchema=True)

26/11/05 18:02:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 18:02:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/05 18:02:42 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/11/05 18:02:42 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/11/05 18:02:42 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/11/05 18:02:42 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/11/05 18:02:42 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/11/05 18:02:42 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
26/11/05 18:02:45 WARN Client: Nei

In [2]:
# Remove duplicates based on the 'customer_id' column
df = df.dropDuplicates(["customer_id"])

In [3]:
# 1. Clean the email column directly (lowercase and remove trailing spaces)
df = df.withColumn("email", lower(trim(df["email"])))

In [4]:
# Trims whitespace from both fields
df = (df
    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name", F.trim(F.col("last_name")))
)


In [5]:
# Replace null values only in the phone_number column
df = df.fillna("Unknown", subset=["phone_number"])

In [6]:
valid_types = ["Offline", "Online", "Both"]
df = df.withColumn("customer_type",F.when(F.col("customer_type").isin(valid_types), F.col("customer_type"))
     .otherwise("Other"))

In [7]:
df.write \
    .mode("overwrite") \
    .saveAsTable("sclsepb4532_staging.customers_stg")

26/11/05 18:04:11 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
